# General odorants and bilateral imaging: Hart, Frank, Lopes et al. (2023), dandiset 000467

Paper: Hart T, Frank DD, Lopes LE et al. "Sparse and stereotyped encoding implicates a core glomerulus
for ant alarm behavior", Cell 186 (2023), doi:10.1016/j.cell.2023.05.025.

Dataset: DANDI:000467, version 0.261004.1655 (doi:10.48324/dandi.000467/0.261004.1655). The paper
cites the dandiset without a version. This is the only published version (published 2026-10-04,
"after adding additional meta data"); the draft has the same 14,685 assets and size, so the most
recent published version is pinned, as the protocol says when the paper cites none.

Claims file: `claims/000467/10.1016-j.cell.2023.05.025.yaml`, written from the accepted author
manuscript in PubMed Central (PMC10334690) with its supplement. Shared methods M1 to M4 are in its
opening comment; the helper `al_imaging.py` says where each parameter comes from.

This notebook holds the general odorant experiment (6 ants, Figure 3F-G) and the bilateral imaging
(3 ants, Figure S4A). The alarm pheromone claims are in `alarm_pheromones.ipynb`.

| Claim | Here |
|---|---|
| fig3f-general-odorants-unique | tested |
| fig3fg-hexanone-broad | tested |
| figs4a-bilateral-symmetry | tested |
| all other claims | `alarm_pheromones.ipynb`, or blocked (see the attempt record) |

In [ ]:
import time

import numpy as np
import pandas as pd
from scipy import ndimage

import al_imaging as al

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
T0 = time.time()
print("dandiset", al.DANDISET_ID, "version", al.DANDISET_VERSION)

In [ ]:
assets = al.list_assets()
trials = al.trials(assets)
alarm_ants = sorted(trials[trials.odor == "heptanone"].ant.unique())
gen_ants = sorted(trials[trials.odor == "3-hexanone"].ant.unique())
bil_ants = sorted(set(trials.ant) - set(alarm_ants) - set(gen_ants))
gen = trials[trials.ant.isin(gen_ants)]
bil = trials[trials.ant.isin(bil_ants)]
print("general odorant ants:", gen_ants)
print("bilateral ants:", bil_ants)

In [ ]:
print("General odorant trials per ant, odor and concentration (planes per trial", sorted(gen.n_planes.unique()),
      ", volumes", sorted(gen.nvol.unique()), "):")
print(gen.groupby(["ant", "odor"]).size().unstack(fill_value=0).to_string())
print(gen.groupby(["odor", "conc"]).size().to_string())
print("\nBilateral trials:")
print(bil.groupby(["ant", "odor", "conc"]).size().unstack(fill_value=0).to_string())
meta = pd.DataFrame([{"ant": ant, **{k: v for k, v in al.file_metadata(trials[trials.ant == ant].iloc[0].blobs[16]).items()
                                     if k in ("shape", "offset", "imaging_rate", "grid_spacing", "age", "location")}}
                     for ant in gen_ants + bil_ants])
print(meta.to_string(index=False))

The general odorant files hold five odorants and the vehicle, the five the paper analyses; the
paper says 7 to 9 odorants were presented, so the others were not deposited. Ethylpyrazine, ethanol,
isopropanol and 3-hexanone were given to all six ants and propionic acid to the four of 2022-10-11.
The bilateral files have a pixel size of 0.578 um (1X zoom), as the paper states, and their location
says "bilateral imaging of both antennal lobes, right is anterior".

In [ ]:
t = time.time()
gmaps = al.trial_maps(gen[gen.nvol == 40])
print(f"general odorant maps: {len(gen)} trials in {time.time() - t:.0f} s")
t = time.time()
bmaps = al.trial_maps(bil)
print(f"bilateral maps: {len(bil)} trials in {time.time() - t:.0f} s")

In [ ]:
for name, mp, tt in (("general", gmaps, gen), ("bilateral", bmaps, bil)):
    sh = pd.Series({i: mp[i]["shift"].max() for i in tt.index})
    print(f"{name}: largest frame shift per trial median {sh.median()} px, max {sh.max()} px; baseline F0 range "
          f"{min(mp[i]['f0'].min() for i in tt.index):.0f} to {max(mp[i]['f0'].max() for i in tt.index):.0f}; "
          f"saturated baseline pixels, max fraction {max(mp[i]['saturated'].max() for i in tt.index):.4f}")
    dups = [(a, b) for ant, g in tt.groupby("ant") for k, a in enumerate(g.index) for b in g.index[k + 1:]
            if np.array_equal(mp[a]["f0"], mp[b]["f0"])]
    print(f"{name}: pairs of trials with identical baseline volumes: {dups if dups else 'none'}")

## fig3f-general-odorants-unique

> Each general odorant activates a unique combination of glomeruli, and the same odorant activates
> similar AL regions in different individuals.

Criterion (the reader's rule): within each ant, the M2 sets of responding ROIs differ between every
pair of the odorants that ant was given; across ants, the centroid of each odorant's responding pixels
(at least 0.2, trial-averaged map), in coordinates scaled to the AL's bounding box in raw
fluorescence, lies closer to the same odorant's centroids in the other ants than to other odorants'
centroids, in more than half of the ant and odorant cases. Reproduced when both hold under every M2
reading. "Closer" is read as: the mean distance to the same odorant's centroids in the other ants is
smaller than the mean distance to each other odorant's centroids in the other ants.

In [ ]:
ODS = ["3-hexanone", "ethylpyrazine", "ethanol", "isopropanol", "propionic acid"]
gmask = {ant: al.al_mask(gmaps, gen[gen.ant == ant].index) for ant in gen_ants}
grois, gresp = {}, {}
rows = []
for ant in gen_ants:
    g = gen[(gen.ant == ant) & gen.odor.isin(ODS)]
    for d in "ab":
        grois[ant, d] = al.draw_rois(gmaps, g, d, gmask[ant])
    for d, v, c in al.READINGS:
        R = {}
        for od in sorted(g.odor.unique()):
            R[od] = al.responding(gmaps, grois[ant, d], g[g.odor == od].index, v, c)[0]
        ods = list(R)
        pairs = [(a, b) for k, a in enumerate(ods) for b in ods[k + 1:]]
        same = [(a, b) for a, b in pairs if np.array_equal(R[a], R[b])]
        rows.append({"ant": ant, "reading": d + v + c, "ROIs": al.n_groups(grois[ant, d]),
                     **{od: int(R[od].sum()) for od in ods}, "pairs": len(pairs), "identical pairs": len(same),
                     "identical": same})
uni = pd.DataFrame(rows)
print(uni.drop(columns=["identical"]).to_string(index=False))
within = uni.groupby("reading")["identical pairs"].sum()
print("\nidentical odorant pairs summed over ants, per reading:", within.to_dict())

# across ants: centroids in AL bounding-box coordinates
cent = []
for ant in gen_ants:
    m = gmask[ant]
    rr, cc = np.where(m)
    box = (rr.min(), rr.max(), cc.min(), cc.max())
    g = gen[gen.ant == ant]
    for od in ODS:
        idx = g[g.odor == od].index
        if not len(idx):
            continue
        mean = np.mean([al._bin(gmaps[i]["proj"], al.BIN) for i in idx], axis=0)
        px = (mean >= al.THRESHOLD) & m
        if not px.any():
            cent.append({"ant": ant, "odor": od, "y": np.nan, "x": np.nan, "n_px": 0})
            continue
        cy, cx = ndimage.center_of_mass(px)
        cent.append({"ant": ant, "odor": od, "y": (cy - box[0]) / (box[1] - box[0]), "x": (cx - box[2]) / (box[3] - box[2]), "n_px": int(px.sum())})
cent = pd.DataFrame(cent)
print(cent.round(3).to_string(index=False))
cases = []
for _, r in cent.dropna().iterrows():
    others = cent[(cent.ant != r.ant)].dropna()
    dist = {od: np.hypot(g.y - r.y, g.x - r.x).mean() for od, g in others.groupby("odor")}
    if r.odor not in dist:
        continue
    cases.append({"ant": r.ant, "odor": r.odor, "same": dist[r.odor], "nearest other": min(v for k, v in dist.items() if k != r.odor),
                  "closer to same": dist[r.odor] < min(v for k, v in dist.items() if k != r.odor)})
cases = pd.DataFrame(cases)
print(cases.round(3).to_string(index=False))
print(f"\nacross ants: {int(cases['closer to same'].sum())} of {len(cases)} ant and odorant cases closer to the same odorant")
print("criterion met (within ants, no identical pair) under readings:", [k for k, v in within.items() if v == 0])

**Result: not reproduced.** Within ants, identical odorant pairs occur under every reading (1 to 9 summed over the 6 ants; bix 7). Across ants, 10 of 27 ant and odorant cases lie closer to the same odorant in the other ants than to any other odorant, fewer than half; 3-hexanone does so in 5 of its 6 ants.

## fig3fg-hexanone-broad

> The breadth of responses varies across general odorants; most activate a few glomeruli, while
> 3-hexanone activates large regions of the ventral and medial AL.

Criterion: breadth is the area of pixels of at least 0.2 in each ant's trial-averaged M1 map.
Reproduced when 3-hexanone has the largest area of the five odorants in every ant where it was
tested, and its median area across ants is at least twice that of each other odorant. The area is
counted inside the AL mask (pixels outside the AL are background).

In [ ]:
area = []
for ant in gen_ants:
    g = gen[gen.ant == ant]
    for od in ODS:
        idx = g[g.odor == od].index
        if not len(idx):
            continue
        mean = np.mean([gmaps[i]["proj"] for i in idx], axis=0)
        mfull = np.kron(gmask[ant], np.ones((al.BIN, al.BIN), bool))
        area.append({"ant": ant, "odor": od, "area_um2": float(((mean >= al.THRESHOLD) & mfull).sum() * al.PIXEL_UM ** 2),
                     "area_all_um2": float((mean >= al.THRESHOLD).sum() * al.PIXEL_UM ** 2)})
area = pd.DataFrame(area)
tab = area.pivot_table(index="ant", columns="odor", values="area_um2").round(0)
print(tab.to_string())
largest = tab.idxmax(axis=1)
print("odorant with the largest area per ant:", largest.to_dict())
med = tab.median()
print("median area (um2):", med.round(0).to_dict())
ratio = {od: round(med["3-hexanone"] / med[od], 2) for od in ODS if od != "3-hexanone"}
print("median 3-hexanone area over each other odorant's median:", ratio)
print("3-hexanone largest in every ant:", bool((largest == "3-hexanone").all()), "; ratios all >= 2:", all(v >= 2 for v in ratio.values()))
print("\nVariant, the whole field of view (no AL mask):")
tab2 = area.pivot_table(index="ant", columns="odor", values="area_all_um2").round(0)
print(tab2.to_string())
print("largest per ant:", tab2.idxmax(axis=1).to_dict(), "; median ratios:",
      {od: round(tab2.median()["3-hexanone"] / tab2.median()[od], 2) for od in ODS if od != "3-hexanone"})

**Result: reproduced.** 3-hexanone has the largest area of at least 0.2 in all 6 ants (median 2,810 um2), 2.57 times the median of ethylpyrazine (1,092 um2) and 7.2 to 51.6 times those of the other three odorants. Without the AL mask it is also the largest in every ant, and the ratio to ethylpyrazine is 2.58.

## figs4a-bilateral-symmetry

> Responses to 4-methyl-3-hexanol are bilaterally symmetrical in the two ALs imaged together.

Criterion (the reader's rule): in each of the three bilateral ants (1X zoom, hexanol 48%), M1 maps split
at the midline between the ALs, one side mirrored. Reproduced when in every ant both ALs have
responding ROIs (M2) and their counts differ by at most one, and the mirrored map of responding pixels
of one AL and the map of the other correlate better than the second does with itself shifted by half
its width. The two ALs lie one above the other in the images, so the midline is a row, placed at the
least fluorescent row between 30% and 70% of the height, and each AL is the largest region of baseline
fluorescence on its side; the mirror is a flip of rows. The files
do not say which AL is the left one.

In [ ]:
PX1X = 0.57773327
rows = []
for ant in bil_ants:
    g = bil[bil.ant == ant]
    from skimage.filters import threshold_otsu
    f0 = np.mean([bmaps[i]["f0"].max(0) for i in g.index], axis=0)
    mask = np.log1p(f0) > threshold_otsu(np.log1p(f0))
    prof = ndimage.uniform_filter1d(np.where(mask, f0, 0).sum(1), 5)
    lo, hi = int(0.3 * len(prof)), int(0.7 * len(prof))
    mid = lo + int(np.argmin(prof[lo:hi]))
    comps = []
    for half in (slice(0, mid), slice(mid, f0.shape[0])):
        part = np.zeros_like(mask)
        part[half] = mask[half]
        lab, n = ndimage.label(part)
        sizes = ndimage.sum(np.ones_like(f0), lab, range(1, n + 1))
        comps.append(ndimage.binary_fill_holes(lab == 1 + int(np.argmax(sizes))))
    sizes = np.array([c.sum() for c in comps])
    w = min(mid, f0.shape[0] - mid)
    hx = g[g.odor == "hexanol"]
    resp_px = np.mean([al._bin(bmaps[i]["proj"], al.BIN) for i in hx.index], axis=0) >= al.THRESHOLD
    top = resp_px[mid - w:mid][::-1]
    bottom = resp_px[mid:mid + w]
    r_mirror = np.corrcoef(top.ravel(), bottom.ravel())[0, 1]
    half = bottom.shape[1] // 2
    r_shift = np.corrcoef(bottom.ravel(), np.roll(bottom, half, axis=1).ravel())[0, 1]
    row = {"ant": ant, "midline row (binned)": mid, "AL areas (bins)": [int(x) for x in sizes],
           "r mirrored": round(r_mirror, 3), "r shifted half width": round(r_shift, 3)}
    for d, v, c in al.READINGS:
        ns = []
        for comp in comps:
            rr = al.draw_rois(bmaps, hx, d, comp, pixel_um=PX1X)
            ns.append(al.responding(bmaps, rr, hx.index, v, c)[1])
        row[d + v + c] = tuple(round(x, 1) for x in ns)
    rows.append(row)
sym = pd.DataFrame(rows)
print(sym.to_string(index=False))
for d, v, c in al.READINGS:
    k = d + v + c
    ok_counts = all(min(x) > 0 and abs(x[0] - x[1]) <= 1 for x in sym[k])
    ok_corr = bool((sym["r mirrored"] > sym["r shifted half width"]).all())
    print(f"reading {k}: counts on both sides and within one in every ant {ok_counts}; mirrored correlation above shifted in every ant {ok_corr}; met {ok_counts and ok_corr}")

**Result: partially reproduced.** In all three ants both ALs have responding ROIs under every reading, and the mirrored maps of responding pixels correlate better (r = 0.072 to 0.185) than the half-width shift (r = -0.024 to -0.008). The ROI counts of the two ALs differ by more than one in at least one ant under every reading (under bix 4 and 5, 4 and 8, 1 and 3).

## Summary

The outcomes of all claims, the differences from the paper's method and the findings about the dataset are summarised in `alarm_pheromones.ipynb` and the attempt record. Here: fig3f-general-odorants-unique: not reproduced; fig3fg-hexanone-broad: reproduced; figs4a-bilateral-symmetry: partially reproduced.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Runtime of this notebook:

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")